# Regime Detection Engine — Walkthrough

**What this notebook does:** Teaches each step in plain English.

| Step | Idea |
|------|------|
| 1 | Download stock prices |
| 2 | Turn prices into clues (returns + volatility) |
| 3 | Train the HMM to find bull / bear / sideways moods |
| 4 | Backtest: does switching by mood beat buy-and-hold? |
| 5 | Charts + optional quantum optimizer |

Run cells **top to bottom**. See `EXPLAINED_SIMPLE.md` for more detail.

## Setup — tell Python where our code lives

Baby terms: import the toolbox we built in `src/regime_engine/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:
    import regime_engine
except ImportError:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from regime_engine.data import fetch_prices, compute_features, train_test_split_by_date
from regime_engine.hmm_detector import RegimeDetector
from regime_engine.sectors import get_preset
from regime_engine.pipeline import run_analysis
from regime_engine.visualize import make_regime_figure, make_equity_figure

%matplotlib inline
print("Ready! Project root:", ROOT)

## Step 1 — Download prices

**Baby terms:** Ask Yahoo Finance "what did these ETFs cost each day?"

We use a **sector preset** — a shortcut bundle of tickers. Try `technology`, `financials`, etc.

In [ ]:
SECTOR = "broad_market"  # try: technology, financials, healthcare, energy
preset = get_preset(SECTOR)
print(f"Preset: {preset.name} — {preset.description}")

tickers = list(dict.fromkeys([preset.primary, *preset.portfolio]))
prices = fetch_prices(tickers, period="10y")
prices.tail()

## Step 2 — Build features (clues for the mood detector)

- **log_return** = how much price moved today (up/down)
- **volatility** = how wild the last ~21 days were

The HMM reads these two numbers each day — not the raw dollar price.

In [ ]:
features = compute_features(prices[preset.primary])
train_feat, test_feat = train_test_split_by_date(features, test_years=2.0)

print(f"Training days: {len(train_feat)} | Test days: {len(test_feat)}")
features.describe()

## Step 3 — Train the HMM (find the hidden moods)

**Baby terms:** The computer studies old days and learns 3 anonymous patterns.
We rename them: highest-return pattern = **bull**, lowest = **bear**, middle = **sideways**.

In [ ]:
detector = RegimeDetector(n_states=3)
detector.fit(train_feat)

train_regimes = detector.predict_regimes(train_feat)
test_regimes = detector.predict_regimes(test_feat)

print("Regime stats (training period):")
display(detector.regime_summary(train_feat, train_regimes))
print("\nTransition matrix (probability of mood changes):")
display(detector.transition_matrix().round(3))

## Step 4 — Full pipeline (strategy + backtest)

**Baby terms:** Pretend we invested by mood on the **test** years (data the model didn't train on).

Compare:
- **Buy & hold** — buy once, do nothing
- **Regime switching** — stocks in bull, bonds in bear, mix in sideways

In [ ]:
result = run_analysis(
    primary_ticker=preset.primary,
    portfolio_tickers=list(preset.portfolio),
    period="10y",
    test_years=2.0,
    use_quantum=False,
)

print(result.report_text)
print(f"\nToday's mood ({result.latest_date}): {result.latest_regime.value}")
print(f"Suggested allocation: {result.current_allocation}")

## Step 5 — Charts

Left: price with colored mood backgrounds (out-of-sample only).
Right: growth of $1 for each strategy.

In [ ]:
fig1 = make_regime_figure(
    result.prices[preset.primary].loc[result.test_features.index],
    result.test_regimes,
    title=f"{preset.primary} regimes (out-of-sample)",
)
fig2 = make_equity_figure(
    [result.buy_hold, result.regime_switch],
    title="Buy & hold vs regime switching",
)
plt.show()

## Bonus — Quantum optimizer (optional, slower)

**Baby terms:** Instead of fixed rules (100% stocks in bull), use QAOA (Qiskit) to search for a good mix per mood.

Set `USE_QUANTUM = True` to try. Falls back to classical math if quantum fails.

In [ ]:
USE_QUANTUM = False  # flip to True to try QAOA

if USE_QUANTUM:
    q_result = run_analysis(
        primary_ticker=preset.primary,
        portfolio_tickers=list(preset.portfolio),
        period="10y",
        test_years=2.0,
        use_quantum=True,
    )
    print(q_result.report_text)
else:
    print("Skipped — set USE_QUANTUM = True to run QAOA.")

## What you learned

1. **data.py** — downloads prices, builds return/vol features
2. **hmm_detector.py** — finds bull/bear/sideways moods
3. **strategies.py** — maps mood → what to own
4. **backtest.py** — simulates past performance honestly (no cheating with future info)
5. **quantum_opt.py** — optional Qiskit portfolio solver

Try the web UI: `streamlit run dashboard.py` from the project folder.